# Cálculo de Volumen y Selección de Depósito GLP



In [1]:
import pandas as pd
import numpy as np

# 1. Parámetros y Constantes (Fuente: Proyecto/Anotaciones/calculo_volumen_deposito.md)
PCS_PROPANO = 13.95  # kWh/kg
DENSIDAD_LIQ = 506.0  # kg/m3 a 20°C
LLENADO_MAX = 0.85
RESERVA_MIN = 0.20
FRACCION_UTIL = LLENADO_MAX - RESERVA_MIN
AUTONOMIA_DIAS = 30  # Requerido por TAREA 3
TEMP_DISENO = -5.0   # IDAE Percentil 99,6% para León
PRESION_SERVICIO = 2.0 # bar (Condición de comprobación solicitada)

print(f"Fracción útil adoptada: {FRACCION_UTIL:.2f}")
print(f"Autonomía de diseño: {AUTONOMIA_DIAS} días")
print(f"Temperatura exterior de diseño: {TEMP_DISENO} ºC")
print(f"Presión de servicio para comprobación: {PRESION_SERVICIO} bar")

Fracción útil adoptada: 0.65
Autonomía de diseño: 30 días
Temperatura exterior de diseño: -5.0 ºC
Presión de servicio para comprobación: 2.0 bar


In [2]:
# 2. Datos de Consumidores (Fuente: Proyecto/Datos.md)
consumidores = [
    {"nombre": "Horno secado 1", "potencia_kw": 60, "horas_dia": 12},
    {"nombre": "Horno secado 2", "potencia_kw": 60, "horas_dia": 12},
    {"nombre": "Caldera vapor", "potencia_kw": 500, "horas_dia": 10},
    {"nombre": "Caldera agua caliente", "potencia_kw": 300, "horas_dia": 8},
    {"nombre": "Horno fusion", "potencia_kw": 700, "horas_dia": 4},
    {"nombre": "Horno decapado", "potencia_kw": 1000, "horas_dia": 6}
]

df_cons = pd.DataFrame(consumidores)
df_cons['energia_diaria_kwh'] = df_cons['potencia_kw'] * df_cons['horas_dia']

energia_total_dia = df_cons['energia_diaria_kwh'].sum()
potencia_max_simultanea = df_cons['potencia_kw'].sum()

print(f"Energía total diaria: {energia_total_dia} kWh/d")
print(f"Potencia máxima simultánea (S=1): {potencia_max_simultanea} kW")

Energía total diaria: 17640 kWh/d
Potencia máxima simultánea (S=1): 2620 kW


In [3]:
# 3. Cálculo de Consumo Másico y Volumétrico
m_dia = energia_total_dia / PCS_PROPANO
v_liq_dia = m_dia / DENSIDAD_LIQ  # m3/día

v_geom_min = (v_liq_dia * AUTONOMIA_DIAS) / FRACCION_UTIL

print(f"Consumo diario: {m_dia:.2f} kg/día")
print(f"Volumen líquido diario: {v_liq_dia:.3f} m3/día")
print(f"--- Volumen geométrico mínimo requerido: {v_geom_min:.2f} m3 ---")

Consumo diario: 1264.52 kg/día
Volumen líquido diario: 2.499 m3/día
--- Volumen geométrico mínimo requerido: 115.34 m3 ---


In [4]:
# 4. Selección del Depósito Comercial (Nueva Configuración por Distancias)
df_dep = pd.read_csv('datos/tabla_caracteristicas_secadores.csv')
df_vap_table = pd.read_csv('datos/caudal_vaporizacion.csv')
df_vapi_table = pd.read_csv('datos/deposito_vaporizador_interno.csv')

# Datos de la nueva selección
dep_grande_ref = 'LP46A-22'
dep_peque_ref = 'LP26A-22'
n_grande = 1
n_peque = 3

v_grande = df_dep[df_dep['Modelo Ref.'] == dep_grande_ref]['Capacidad nominal (litros)'].values[0]
v_peque = df_dep[df_dep['Modelo Ref.'] == dep_peque_ref]['Capacidad nominal (litros)'].values[0]
v_total = (v_grande * n_grande) + (v_peque * n_peque)

print(f"Configuración de Almacenamiento: {n_grande} x {dep_grande_ref} + {n_peque} x {dep_peque_ref}")
print(f"Capacidad total: {v_total} litros")

Configuración de Almacenamiento: 1 x LP46A-22 + 3 x LP26A-22
Capacidad total: 125100 litros


In [5]:
# 5. Verificación de Vaporización y Selección de Vaporizador Forzado
caudal_nec_kgh = potencia_max_simultanea / PCS_PROPANO
print(f"Demanda punta necesaria: {caudal_nec_kgh:.2f} kg/h")

# 5.1. Comprobación de Vaporización Natural (al 20% de llenado)
vol_grande_m3 = 46.2
vol_peque_m3 = 26.3

vap_grande = df_vap_table[(abs(df_vap_table['Volum. m3'] - vol_grande_m3) < 0.01) & (df_vap_table['Pres. bar'] == PRESION_SERVICIO)]
vap_peque = df_vap_table[(abs(df_vap_table['Volum. m3'] - vol_peque_m3) < 0.01) & (df_vap_table['Pres. bar'] == PRESION_SERVICIO)]

q_nat_grande = vap_grande['Caudal Aéreo -5°C'].values[0]
q_nat_peque_total = vap_peque['Caudal Aéreo -5°C'].values[0] * n_peque
q_nat_total = q_nat_grande + q_nat_peque_total

print(f"Vaporización natural {dep_grande_ref}: {q_nat_grande} kg/h")
print(f"Vaporización natural {dep_peque_ref} ({n_peque} uds): {q_nat_peque_total:.1f} kg/h")
print(f"Vaporización natural total (-5ºC, 2 bar, 20% llenado): {q_nat_total:.1f} kg/h")

if q_nat_total < caudal_nec_kgh:
    deficit = caudal_nec_kgh - q_nat_total
    print(f"Déficit de vaporización natural: {deficit:.2f} kg/h")
    
    # 5.2. Selección de Vaporizador Interno
    # Solo se requiere en el depósito más grande para cubrir el déficit total.
    capacidades_vapi = {"VIA 150": 150, "VIA 300": 300, "VIB 500": 500}
    potencias_caldera = {"VIA 150": 17.5, "VIA 300": 35, "VIB 500": 58}
    
    vapi_seleccionado = None
    for mod, cap in capacidades_vapi.items():
        if cap >= deficit:
            vapi_seleccionado = mod
            break
    
    if vapi_seleccionado:
        print(f"\n--- Selección de Vaporización Forzada ---")
        print(f"Vaporizador Interno Seleccionado (en {dep_grande_ref.replace('LP', 'LPVI')}): {vapi_seleccionado}")
        print(f"Capacidad forzada: {capacidades_vapi[vapi_seleccionado]} kg/h")
        print(f"Potencia de caldera requerida: {potencias_caldera[vapi_seleccionado]} kW")
        print(f"Vaporización Total (Nat + Forz): {q_nat_total + capacidades_vapi[vapi_seleccionado]:.2f} kg/h")
        print(f"RESULTADO: Suministro garantizado mediante sistema mixto (Vaporizador en el depósito de {vol_grande_m3}m3).")
    else:
        print("No se encontró un vaporizador interno suficiente.")
else:
    print("Vaporización natural suficiente.")

Demanda punta necesaria: 187.81 kg/h
Vaporización natural LP46A-22: 47.0 kg/h
Vaporización natural LP26A-22 (3 uds): 84.3 kg/h
Vaporización natural total (-5ºC, 2 bar, 20% llenado): 131.3 kg/h
Déficit de vaporización natural: 56.51 kg/h

--- Selección de Vaporización Forzada ---
Vaporizador Interno Seleccionado (en LPVI46A-22): VIA 150
Capacidad forzada: 150 kg/h
Potencia de caldera requerida: 17.5 kW
Vaporización Total (Nat + Forz): 281.30 kg/h
RESULTADO: Suministro garantizado mediante sistema mixto (Vaporizador en el depósito de 46.2m3).


## 6. Potencia termica del armario de calefaccion

El armario de calefaccion aporta el calor al circuito cerrado de agua que alimenta el serpentín interno del vaporizador forzado. Por tanto, su potencia minima se fija por la potencia de caldera requerida por el modelo de vaporizador seleccionado.

In [6]:
# 6. Calculo de potencia termica del armario de calefaccion
armario_calefaccion = 'VPC30C'
potencia_nominal_armario_kw = 45.0  # kW, caldera integrada en VPC30C según ficha tecnica consultada en GLP

if 'vapi_seleccionado' in globals() and vapi_seleccionado:
    capacidad_forzada_kgh = capacidades_vapi[vapi_seleccionado]
    potencia_minima_armario_kw = potencias_caldera[vapi_seleccionado]
    margen_vaporizacion_kgh = (q_nat_total + capacidad_forzada_kgh) - caudal_nec_kgh
    margen_potencia_armario_kw = potencia_nominal_armario_kw - potencia_minima_armario_kw

    print('--- Potencia termica del armario de calefaccion ---')
    print(f'Demanda punta de GLP: {caudal_nec_kgh:.2f} kg/h')
    print(f'Vaporizacion natural disponible: {q_nat_total:.1f} kg/h')
    print(f'Deficit que obliga a vaporizacion forzada: {deficit:.2f} kg/h')
    print(f'Vaporizador interno seleccionado: {vapi_seleccionado}')
    print(f'Capacidad forzada del vaporizador: {capacidad_forzada_kgh:.0f} kg/h')
    print(f'Armario de calefaccion asociado: {armario_calefaccion}')
    print(f'Potencia nominal del armario: {potencia_nominal_armario_kw:.1f} kW')
    print(f'Potencia termica minima requerida: {potencia_minima_armario_kw:.1f} kW')
    print(f'Margen de potencia del armario: {margen_potencia_armario_kw:.1f} kW')
    print(f'Margen total de vaporizacion: {margen_vaporizacion_kgh:.2f} kg/h')
    print(f'Comprobacion: {potencia_nominal_armario_kw:.1f} kW >= {potencia_minima_armario_kw:.1f} kW. Armario suficiente.')
else:
    potencia_minima_armario_kw = 0.0
    print('No se requiere potencia de armario: la vaporizacion natural resulta suficiente.')

--- Potencia termica del armario de calefaccion ---
Demanda punta de GLP: 187.81 kg/h
Vaporizacion natural disponible: 131.3 kg/h
Deficit que obliga a vaporizacion forzada: 56.51 kg/h
Vaporizador interno seleccionado: VIA 150
Capacidad forzada del vaporizador: 150 kg/h
Armario de calefaccion asociado: VPC30C
Potencia nominal del armario: 45.0 kW
Potencia termica minima requerida: 17.5 kW
Margen de potencia del armario: 27.5 kW
Margen total de vaporizacion: 93.49 kg/h
Comprobacion: 45.0 kW >= 17.5 kW. Armario suficiente.


## 7. Dimensionado de tuberías GLP

Se ejecuta el dimensionado por tramos con Renouard en media presión, longitud de cálculo iterativa por accesorios y verificación de velocidad. La memoria técnica se genera como `calculos/dimensionado_tuberias_glp.md`.


In [7]:
# 7.1. Ejecución del dimensionado por tramos en la propia libreta
from pathlib import Path
from math import sqrt
from functools import lru_cache

P_ATM_BAR = 1.01325
P_INICIAL_REL_BAR = 1.70
P_INICIAL_ABS_BAR = P_INICIAL_REL_BAR + P_ATM_BAR
CAIDA_MAXIMA_FRACCION = 0.05
P_MIN_REL_BAR = P_INICIAL_REL_BAR * (1.0 - CAIDA_MAXIMA_FRACCION)
P_MIN_ABS_BAR = P_MIN_REL_BAR + P_ATM_BAR
DC_PROPANO = 1.16
DENSIDAD_PROPANO_GAS_KG_M3 = 1.882
VEL_MAX_MS = 20.0
NOTEBOOK_GLP_ID = '3bae2f27-6ddf-4f0c-99bf-f37c4e05dea3'
BASE_CALCULOS = Path('.').resolve()
RAIZ_PROYECTO = BASE_CALCULOS.parent
SALIDA_MARKDOWN = BASE_CALCULOS / 'dimensionado_tuberias_glp.md'
SALIDA_ANOTACION = RAIZ_PROYECTO / 'Proyecto' / 'Anotaciones' / 'dimensionado_tuberias_glp.md'

consulta_nlm_materiales = '''Consulta ejecutada con CLI: nlm notebook query 3bae2f27-6ddf-4f0c-99bf-f37c4e05dea3 "Para una instalacion aerea exterior de GLP en media presion, que materiales de tuberia son admisibles/recomendables, que limitaciones aplican a cobre, acero y PE, y que criterio debe usarse para justificar el material seleccionado."

Resultado resumido: para instalacion aerea de GLP en media presion se citan cobre duro estirado sin soldadura y acero como materiales usados en la practica. El PE queda descartado en canalizaciones aereas por degradacion frente a radiacion UV y se reserva a tramos enterrados. El acero requiere proteccion pasiva frente a corrosion y, segun las fuentes consultadas, puede quedar condicionado a autorizacion expresa en redes de media presion. El cobre en instalacion aerea debe garantizar espesor minimo de 1 mm. El criterio resistente se justifica comprobando que la presion maxima de servicio admisible del material sea suficiente frente a la presion de servicio.'''

material_seleccionado = 'Cobre duro estirado sin soldadura EN 1057, espesor minimo 1 mm, instalacion aerea protegida'
justificacion_material = (
    'Se selecciona cobre duro estirado sin soldadura porque la consulta documental lo identifica como material practico admisible '
    'para fase gas en GLP y evita la restriccion indicada para redes de acero en media presion salvo autorizacion. '
    'El PE se descarta por ser canalizacion aerea exterior. La alternativa de acero al carbono queda tecnicamente posible '
    'solo con proteccion pasiva y autorizacion o documentacion especifica.'
)

catalogo_diametros = pd.DataFrame([
    {'material': 'cobre_duro_en1057', 'designacion': '15x1', 'D_int_mm': 13.0, 'espesor_mm': 1.0},
    {'material': 'cobre_duro_en1057', 'designacion': '18x1', 'D_int_mm': 16.0, 'espesor_mm': 1.0},
    {'material': 'cobre_duro_en1057', 'designacion': '22x1', 'D_int_mm': 20.0, 'espesor_mm': 1.0},
    {'material': 'cobre_duro_en1057', 'designacion': '28x1', 'D_int_mm': 26.0, 'espesor_mm': 1.0},
    {'material': 'cobre_duro_en1057', 'designacion': '35x1.5', 'D_int_mm': 32.0, 'espesor_mm': 1.5},
    {'material': 'cobre_duro_en1057', 'designacion': '42x1.5', 'D_int_mm': 39.0, 'espesor_mm': 1.5},
    {'material': 'cobre_duro_en1057', 'designacion': '54x2', 'D_int_mm': 50.0, 'espesor_mm': 2.0},
    {'material': 'cobre_duro_en1057', 'designacion': '64x2', 'D_int_mm': 60.0, 'espesor_mm': 2.0},
    {'material': 'cobre_duro_en1057', 'designacion': '76.1x2', 'D_int_mm': 72.1, 'espesor_mm': 2.0},
    {'material': 'cobre_duro_en1057', 'designacion': '88.9x2', 'D_int_mm': 84.9, 'espesor_mm': 2.0},
    {'material': 'cobre_duro_en1057', 'designacion': '108x2.5', 'D_int_mm': 103.0, 'espesor_mm': 2.5},
]).sort_values('D_int_mm').reset_index(drop=True)

coef_accesorios = {'codo_90': 30, 'codo_45': 15, 'te_linea': 20, 'te_desviada': 60, 'valvula_bola': 10, 'reduccion': 10}
accesorios_base = {
    'D1-D4': {'codo_90': 1, 'valvula_bola': 1}, 'D2-D4': {'codo_90': 1, 'valvula_bola': 1},
    'D3-D4': {'codo_90': 1, 'valvula_bola': 1}, 'D4 vertical': {'codo_90': 1, 'valvula_bola': 1},
    'D4-A': {'codo_90': 1, 'valvula_bola': 1, 'te_linea': 1},
    'A-C1': {'te_desviada': 1, 'valvula_bola': 1, 'reduccion': 1}, 'A-B': {'codo_90': 1, 'te_linea': 1},
    'B-C2': {'te_desviada': 1, 'valvula_bola': 1, 'reduccion': 1}, 'B-C': {'codo_90': 1, 'te_linea': 1},
    'C-C3': {'te_desviada': 1, 'valvula_bola': 1, 'reduccion': 1}, 'C-D': {'codo_90': 1, 'te_linea': 1},
    'D-C4': {'te_desviada': 1, 'valvula_bola': 1, 'reduccion': 1}, 'D-E': {'codo_90': 1, 'te_linea': 1},
    'E-C5': {'te_desviada': 1, 'valvula_bola': 1, 'reduccion': 1}, 'E-C6': {'te_desviada': 1, 'valvula_bola': 1, 'reduccion': 1},
}

def tabla_md(df, cols=None, floatfmt='.3f'):
    data = df.copy() if cols is None else df[cols].copy()
    for col in data.select_dtypes(include=['float', 'float64']).columns:
        data[col] = data[col].map(lambda x: '' if pd.isna(x) else format(x, floatfmt))
    data = data.fillna('').astype(str)
    headers = list(data.columns)
    rows = data.values.tolist()
    def clean(value):
        return value.replace('|', '\\|').replace('\n', ' ')
    return '\n'.join([
        '| ' + ' | '.join(clean(h) for h in headers) + ' |',
        '| ' + ' | '.join('---' for _ in headers) + ' |',
        *['| ' + ' | '.join(clean(v) for v in row) + ' |' for row in rows],
    ])

def longitud_calculo(L_real_m, D_mm, accesorios):
    total = float(L_real_m)
    for nombre, cantidad in accesorios.items():
        total += cantidad * coef_accesorios[nombre] * (D_mm / 1000.0)
    return total

def renouard_mp(PA_abs_bar, Q_m3_h, D_mm, Lc_m, dc=DC_PROPANO):
    termino = 51.5 * dc * Lc_m * (Q_m3_h ** 1.82) / (D_mm ** 4.82)
    pb2 = PA_abs_bar ** 2 - termino
    if pb2 <= 0:
        return None, None, termino
    PB_abs_bar = sqrt(pb2)
    return PB_abs_bar, PA_abs_bar - PB_abs_bar, termino

def velocidad_gas(Q_m3_h, P_abs_bar, D_mm):
    return 378.04 * Q_m3_h / (P_abs_bar * (D_mm ** 2))

tramos_csv = pd.read_csv('datos/longitudes_Tramos.csv', sep=';').dropna(how='all').copy()
tramos_csv = tramos_csv.rename(columns={'Nº': 'numero', 'Tramo / zona': 'tramo_zona', 'Designación': 'designacion', 'Longitud (m)': 'longitud_m'})
tramos_csv['numero'] = tramos_csv['numero'].astype(int)
tramos_csv['longitud_m'] = pd.to_numeric(tramos_csv['longitud_m'], errors='coerce')
tramos_csv['designacion'] = tramos_csv['designacion'].astype(str).str.strip()
tramos_red = tramos_csv[tramos_csv['numero'] != 1].copy()
tramos_red[['nodo_ini', 'nodo_fin']] = tramos_red['designacion'].str.split('-', expand=True)
tramos_red['tipo_tramo'] = 'red_principal'
depositos_verticales = pd.DataFrame([
    {'numero': 1, 'tramo_zona': 'Derivacion vertical deposito D1', 'designacion': 'D1-D4', 'longitud_m': 1.90, 'nodo_ini': 'D1', 'nodo_fin': 'D4', 'tipo_tramo': 'derivacion_deposito'},
    {'numero': 1, 'tramo_zona': 'Derivacion vertical deposito D2', 'designacion': 'D2-D4', 'longitud_m': 1.90, 'nodo_ini': 'D2', 'nodo_fin': 'D4', 'tipo_tramo': 'derivacion_deposito'},
    {'numero': 1, 'tramo_zona': 'Derivacion vertical deposito D3', 'designacion': 'D3-D4', 'longitud_m': 1.90, 'nodo_ini': 'D3', 'nodo_fin': 'D4', 'tipo_tramo': 'derivacion_deposito'},
    {'numero': 1, 'tramo_zona': 'Derivacion vertical deposito D4', 'designacion': 'D4 vertical', 'longitud_m': 1.90, 'nodo_ini': 'D4_dep', 'nodo_fin': 'D4', 'tipo_tramo': 'derivacion_deposito'},
])
df_tramos_limpio = pd.concat([depositos_verticales, tramos_red], ignore_index=True)
df_tramos_limpio = df_tramos_limpio[['numero', 'tramo_zona', 'designacion', 'longitud_m', 'nodo_ini', 'nodo_fin', 'tipo_tramo']]

df_consumidores_red = df_cons.copy().reset_index(drop=True)
df_consumidores_red['nodo'] = ['C1', 'C2', 'C3', 'C4', 'C5', 'C6']
df_consumidores_red['q_kg_h'] = df_consumidores_red['potencia_kw'] / PCS_PROPANO
df_consumidores_red['q_m3_h'] = df_consumidores_red['q_kg_h'] / DENSIDAD_PROPANO_GAS_KG_M3
q_total_kg_h = df_consumidores_red['q_kg_h'].sum()
q_total_m3_h = df_consumidores_red['q_m3_h'].sum()
children = {}
for _, row in tramos_red.iterrows():
    children.setdefault(row['nodo_ini'], []).append(row['nodo_fin'])
consumo_por_nodo = df_consumidores_red.set_index('nodo')['q_m3_h'].to_dict()
@lru_cache(None)
def caudal_descendente(nodo):
    return consumo_por_nodo.get(nodo, 0.0) + sum(caudal_descendente(hijo) for hijo in children.get(nodo, []))
q_por_designacion = {row['designacion']: caudal_descendente(row['nodo_fin']) for _, row in tramos_red.iterrows()}
for designacion in depositos_verticales['designacion']:
    q_por_designacion[designacion] = q_total_m3_h / len(depositos_verticales)
df_tramos_limpio['q_m3_h'] = df_tramos_limpio['designacion'].map(q_por_designacion)
df_tramos_limpio['q_kg_h'] = df_tramos_limpio['q_m3_h'] * DENSIDAD_PROPANO_GAS_KG_M3

filas_acc = []
for designacion, accesorios in accesorios_base.items():
    for accesorio, cantidad in accesorios.items():
        filas_acc.append({'designacion': designacion, 'accesorio': accesorio, 'cantidad': cantidad, 'coef_leq_D': coef_accesorios[accesorio], 'criterio': 'visible/hipotesis conservadora segun esquema_instalacion.pdf'})
df_accesorios = pd.DataFrame(filas_acc)

def dimensionar_tramo(tramo, P_ini_abs_bar, catalogo):
    accesorios = accesorios_base.get(tramo['designacion'], {})
    Q = float(tramo['q_m3_h'])
    candidatos = []
    for _, tuberia in catalogo.iterrows():
        D = float(tuberia['D_int_mm'])
        Lc = longitud_calculo(tramo['longitud_m'], D, accesorios)
        q_d = Q / D if D else float('inf')
        P_fin_abs, dp_bar, _ = renouard_mp(P_ini_abs_bar, Q, D, Lc)
        if P_fin_abs is None:
            fila = {'designacion_tubo': tuberia['designacion'], 'D_int_mm': D, 'Lc_m': Lc, 'Q_D': q_d, 'P_fin_abs_bar': np.nan, 'P_fin_rel_bar': np.nan, 'delta_p_bar': np.nan, 'velocidad_ms': np.nan, 'cumple_Q_D': q_d < 150, 'cumple_presion': False, 'cumple_velocidad': False, 'estado': 'No conforme: presion final no real'}
        else:
            v = velocidad_gas(Q, P_fin_abs, D)
            cumple_qd = q_d < 150
            cumple_p = P_fin_abs >= P_MIN_ABS_BAR
            cumple_v = v <= VEL_MAX_MS
            fila = {'designacion_tubo': tuberia['designacion'], 'D_int_mm': D, 'Lc_m': Lc, 'Q_D': q_d, 'P_fin_abs_bar': P_fin_abs, 'P_fin_rel_bar': P_fin_abs - P_ATM_BAR, 'delta_p_bar': dp_bar, 'velocidad_ms': v, 'cumple_Q_D': cumple_qd, 'cumple_presion': cumple_p, 'cumple_velocidad': cumple_v, 'estado': 'Conforme' if (cumple_qd and cumple_p and cumple_v) else 'No conforme'}
        candidatos.append(fila)
        if fila['estado'] == 'Conforme':
            return fila, pd.DataFrame(candidatos)
    fallo = candidatos[-1].copy()
    fallo['estado'] = 'No conforme: ningun diametro del catalogo cumple'
    return fallo, pd.DataFrame(candidatos)

orden_depositos = ['D1-D4', 'D2-D4', 'D3-D4', 'D4 vertical']
orden_red = ['D4-A', 'A-C1', 'A-B', 'B-C2', 'B-C', 'C-C3', 'C-D', 'D-C4', 'D-E', 'E-C5', 'E-C6']
presion_nodo = {'D4': P_INICIAL_ABS_BAR}
resultados = []
candidatos_por_tramo = {}
for designacion in orden_depositos:
    tramo = df_tramos_limpio[df_tramos_limpio['designacion'] == designacion].iloc[0].to_dict()
    elegido, candidatos = dimensionar_tramo(tramo, P_INICIAL_ABS_BAR, catalogo_diametros)
    candidatos_por_tramo[designacion] = candidatos
    resultados.append({**tramo, **elegido, 'P_ini_abs_bar': P_INICIAL_ABS_BAR, 'P_ini_rel_bar': P_INICIAL_REL_BAR})
for designacion in orden_red:
    tramo = df_tramos_limpio[df_tramos_limpio['designacion'] == designacion].iloc[0].to_dict()
    P_ini = presion_nodo[tramo['nodo_ini']]
    elegido, candidatos = dimensionar_tramo(tramo, P_ini, catalogo_diametros)
    candidatos_por_tramo[designacion] = candidatos
    resultados.append({**tramo, **elegido, 'P_ini_abs_bar': P_ini, 'P_ini_rel_bar': P_ini - P_ATM_BAR})
    if not np.isnan(elegido.get('P_fin_abs_bar', np.nan)):
        presion_nodo[tramo['nodo_fin']] = elegido['P_fin_abs_bar']
columnas_dimensionado = ['numero', 'tipo_tramo', 'tramo_zona', 'designacion', 'nodo_ini', 'nodo_fin', 'longitud_m', 'q_m3_h', 'q_kg_h', 'designacion_tubo', 'D_int_mm', 'Lc_m', 'Q_D', 'P_ini_rel_bar', 'P_fin_rel_bar', 'delta_p_bar', 'velocidad_ms', 'cumple_Q_D', 'cumple_presion', 'cumple_velocidad', 'estado']
df_dimensionado = pd.DataFrame(resultados)[columnas_dimensionado].copy()
df_no_conformidades = df_dimensionado[df_dimensionado['estado'] != 'Conforme'].copy()
resumen_depositos = df_dimensionado[df_dimensionado['tipo_tramo'] == 'derivacion_deposito'].copy()
resumen_depositos_agrupado = pd.DataFrame([{'designacion': 'D1/D2/D3/D4 verticales', 'unidades': len(resumen_depositos), 'longitud_m_por_unidad': resumen_depositos['longitud_m'].iloc[0], 'q_m3_h_por_unidad': resumen_depositos['q_m3_h'].iloc[0], 'designacion_tubo': resumen_depositos['designacion_tubo'].mode().iloc[0], 'D_int_mm': resumen_depositos['D_int_mm'].max(), 'Lc_m_por_unidad': resumen_depositos['Lc_m'].max(), 'P_fin_rel_bar_min': resumen_depositos['P_fin_rel_bar'].min(), 'velocidad_ms_max': resumen_depositos['velocidad_ms'].max(), 'estado': 'Conforme' if (resumen_depositos['estado'] == 'Conforme').all() else 'No conforme'}])
tramo_control = df_dimensionado[df_dimensionado['designacion'] == 'D4-A'].iloc[0]
control_manual = {'tramo': tramo_control['designacion'], 'Lc_recalculada_m': longitud_calculo(tramo_control['longitud_m'], tramo_control['D_int_mm'], accesorios_base[tramo_control['designacion']]), 'Q_D_recalculado': tramo_control['q_m3_h'] / tramo_control['D_int_mm'], 'velocidad_recalculada_ms': velocidad_gas(tramo_control['q_m3_h'], tramo_control['P_fin_rel_bar'] + P_ATM_BAR, tramo_control['D_int_mm'])}

def texto_dimensionado_markdown(titulo, contexto):
    cols_resultados = ['designacion', 'tipo_tramo', 'longitud_m', 'q_m3_h', 'designacion_tubo', 'D_int_mm', 'Lc_m', 'Q_D', 'P_ini_rel_bar', 'P_fin_rel_bar', 'delta_p_bar', 'velocidad_ms', 'estado']
    cols_consumidores = ['nodo', 'nombre', 'potencia_kw', 'q_kg_h', 'q_m3_h']
    cols_acc = ['designacion', 'accesorio', 'cantidad', 'coef_leq_D', 'criterio']
    no_conf_text = 'No se detectan no conformidades.' if df_no_conformidades.empty else tabla_md(df_no_conformidades[cols_resultados])
    return f'''# {titulo}\n\n{contexto}\n\n## Fuentes y criterios\n\n- Longitudes: `calculos/datos/longitudes_Tramos.csv`.\n- Esquema: `esquema_instalacion.pdf`.\n- Formulas: `Proyecto/skills/doc_tecnica_glp/references/formulario.md` y `Proyecto/Especificaciones/metodologia-longitud-calculo.md`.\n- Consulta NotebookLM: notebook `GLP` (`{NOTEBOOK_GLP_ID}`) mediante CLI `nlm notebook query`.\n\n{consulta_nlm_materiales}\n\n## Hipotesis de calculo\n\n- Gas de referencia: propano.\n- Densidad corregida Renouard: `dc = {DC_PROPANO}`.\n- PCS usado para pasar de potencia a caudal masico: `{PCS_PROPANO:.2f} kWh/kg`.\n- Densidad de propano gas para obtener caudal volumetrico: `{DENSIDAD_PROPANO_GAS_KG_M3:.3f} kg/m3`.\n- Presion inicial de red: `{P_INICIAL_REL_BAR:.2f} bar relativos`.\n- Caida maxima admisible total: `{CAIDA_MAXIMA_FRACCION * 100:.1f}%`, equivalente a presion minima `{P_MIN_REL_BAR:.3f} bar relativos`.\n- Velocidad maxima admisible: `{VEL_MAX_MS:.1f} m/s`.\n- La fila 1 del CSV se trata como cuatro derivaciones verticales de deposito de 1,90 m y se resume de forma agrupada.\n- Los accesorios no confirmados visualmente se incorporan mediante hipotesis conservadora documentada.\n\n## Material seleccionado\n\n{justificacion_material}\n\nMaterial adoptado: **{material_seleccionado}**.\n\n### Catalogo de diametros\n\n{tabla_md(catalogo_diametros)}\n\n## Formulas empleadas\n\nConversion de potencia a caudal:\n\n```text\nQ_kg/h = P_kW / PCS_propano\nQ_m3/h = Q_kg/h / rho_propano_gas\n```\n\nLongitud de calculo iterativa por accesorios:\n\n```text\nLc(D) = Lreal + sum(n_i * coef_i * D_mm / 1000)\n```\n\nRenouard para media presion:\n\n```text\nPA_abs^2 - PB_abs^2 = 51.5 * dc * Lc * Q^1.82 / D^4.82\n```\n\nVelocidad del gas:\n\n```text\nv = 378.04 * Q / (P_abs * D^2)\n```\n\nCondicion de aplicacion y aceptacion:\n\n```text\nQ / D < 150\nPB_rel >= {P_MIN_REL_BAR:.3f} bar\nv <= {VEL_MAX_MS:.1f} m/s\n```\n\n## Consumidores y caudales\n\n{tabla_md(df_consumidores_red[cols_consumidores])}\n\n## Tramos modelizados\n\n{tabla_md(df_tramos_limpio[['designacion', 'tipo_tramo', 'nodo_ini', 'nodo_fin', 'longitud_m', 'q_m3_h', 'q_kg_h']])}\n\n## Inventario de accesorios\n\n{tabla_md(df_accesorios[cols_acc])}\n\n## Resumen de derivaciones verticales de deposito\n\n{tabla_md(resumen_depositos_agrupado)}\n\n## Resultados de dimensionado\n\n{tabla_md(df_dimensionado[cols_resultados])}\n\n## No conformidades\n\n{no_conf_text}\n\n## Control manual de trazabilidad\n\nTramo de control: `{control_manual['tramo']}`.\n\n- `Lc` recalculada: `{control_manual['Lc_recalculada_m']:.3f} m`.\n- `Q/D` recalculado: `{control_manual['Q_D_recalculado']:.3f}`.\n- Velocidad recalculada: `{control_manual['velocidad_recalculada_ms']:.3f} m/s`.\n\n## Limitaciones\n\n- El inventario de accesorios se basa en el esquema y en hipotesis conservadoras para los elementos no distinguibles con certeza en el PDF.\n- El catalogo de diametros se fija como catalogo de calculo para este proyecto; debe contrastarse con proveedor antes de mediciones o presupuesto definitivo.\n- Si se sustituye el cobre por acero, debe documentarse autorizacion o criterio equivalente para media presion y proteccion pasiva frente a corrosion.\n\n## Conclusion tecnica\n\nCon las hipotesis adoptadas, la red queda dimensionada para el caudal punta simultaneo de `{q_total_kg_h:.2f} kg/h` (`{q_total_m3_h:.2f} m3/h`). Todos los tramos modelizados cumplen la condicion de Renouard `Q/D < 150`, la velocidad maxima de `{VEL_MAX_MS:.1f} m/s` y la presion minima relativa de `{P_MIN_REL_BAR:.3f} bar`. La solucion queda condicionada a validar en fase de detalle el inventario definitivo de accesorios y el catalogo comercial exacto del tubo seleccionado.\n'''

SALIDA_MARKDOWN.write_text(texto_dimensionado_markdown('Dimensionado de tuberias GLP por tramos', 'Documento generado desde `calculos/calculos.ipynb` como salida reproducible del calculo.'), encoding='utf-8')
SALIDA_ANOTACION.parent.mkdir(parents=True, exist_ok=True)
SALIDA_ANOTACION.write_text(texto_dimensionado_markdown('Anotacion tecnica: dimensionado de tuberias GLP', 'Esta anotacion documenta el criterio tecnico y matematico empleado para dimensionar la red de distribucion de GLP del Grupo G1-1. Debe usarse como soporte documental principal del apartado de red de distribucion de la memoria.'), encoding='utf-8')
resultados_dimensionado = {'salida_markdown': SALIDA_MARKDOWN, 'salida_anotacion': SALIDA_ANOTACION, 'q_total_kg_h': q_total_kg_h, 'q_total_m3_h': q_total_m3_h, 'material_seleccionado': material_seleccionado, 'justificacion_material': justificacion_material, 'consulta_nlm_materiales': consulta_nlm_materiales}
print(f'Material seleccionado: {material_seleccionado}')
print(f'Caudal total: {q_total_m3_h:.2f} m3/h ({q_total_kg_h:.2f} kg/h)')
print(f'Markdown de cálculo generado: {SALIDA_MARKDOWN}')
print(f'Anotación técnica generada: {SALIDA_ANOTACION}')
print(f'Tramos dimensionados: {len(df_dimensionado)}')
print(f'No conformidades: {len(df_no_conformidades)}')


Material seleccionado: Cobre duro estirado sin soldadura EN 1057, espesor minimo 1 mm, instalacion aerea protegida
Caudal total: 99.79 m3/h (187.81 kg/h)
Markdown de cálculo generado: H:\Unidades compartidas\Practicas_Inst2\4_GLPs\calculos\dimensionado_tuberias_glp.md
Anotación técnica generada: H:\Unidades compartidas\Practicas_Inst2\4_GLPs\Proyecto\Anotaciones\dimensionado_tuberias_glp.md
Tramos dimensionados: 15
No conformidades: 0


In [8]:
# 7.2. Tabla limpia de tramos y caudales
df_tramos_limpio


,numero,tramo_zona,designacion,longitud_m,nodo_ini,nodo_fin,tipo_tramo,q_m3_h,q_kg_h
0,1,Derivacion vertical deposito D1,D1-D4,1.90,D1,D4,derivacion_deposito,24.948674,46.953405
1,1,Derivacion vertical deposito D2,D2-D4,1.90,D2,D4,derivacion_deposito,24.948674,46.953405
2,1,Derivacion vertical deposito D3,D3-D4,1.90,D3,D4,derivacion_deposito,24.948674,46.953405
3,1,Derivacion vertical deposito D4,D4 vertical,1.90,D4_dep,D4,derivacion_deposito,24.948674,46.953405
4,2,Colector superior desde zona de depósitos,D4-A,15.12,D4,A,red_principal,99.794697,187.813620
5,3,Derivación hacia C1,A-C1,1.81,A,C1,red_principal,2.285375,4.301075
6,4,Bajante a C2,A-B,4.97,A,B,red_principal,97.509322,183.512545
7,5,Derivación hacia C2,B-C2,21.79,B,C2,red_principal,2.285375,4.301075
8,6,Bajante a C3,B-C,17.32,B,C,red_principal,95.223948,179.211470
9,7,Derivación hacia C3,C-C3,1.75,C,C3,red_principal,19.044790,35.842294


In [9]:
# 7.3. Consumidores asociados a C1-C6 y caudales de cálculo
df_consumidores_red


,nombre,potencia_kw,horas_dia,energia_diaria_kwh,nodo,q_kg_h,q_m3_h
0,Horno secado 1,60,12,720,C1,4.301075,2.285375
1,Horno secado 2,60,12,720,C2,4.301075,2.285375
2,Caldera vapor,500,10,5000,C3,35.842294,19.044790
3,Caldera agua caliente,300,8,2400,C4,21.505376,11.426874
4,Horno fusion,700,4,2800,C5,50.179211,26.662705
5,Horno decapado,1000,6,6000,C6,71.684588,38.089579


In [10]:
# 7.4. Inventario de accesorios por tramo
df_accesorios


,designacion,accesorio,cantidad,coef_leq_D,criterio
0,D1-D4,codo_90,1,30,visible/hipotesis conservadora segun esquema_i...
1,D1-D4,valvula_bola,1,10,visible/hipotesis conservadora segun esquema_i...
2,D2-D4,codo_90,1,30,visible/hipotesis conservadora segun esquema_i...
3,D2-D4,valvula_bola,1,10,visible/hipotesis conservadora segun esquema_i...
4,D3-D4,codo_90,1,30,visible/hipotesis conservadora segun esquema_i...
5,D3-D4,valvula_bola,1,10,visible/hipotesis conservadora segun esquema_i...
6,D4 vertical,codo_90,1,30,visible/hipotesis conservadora segun esquema_i...
7,D4 vertical,valvula_bola,1,10,visible/hipotesis conservadora segun esquema_i...
8,D4-A,codo_90,1,30,visible/hipotesis conservadora segun esquema_i...
9,D4-A,valvula_bola,1,10,visible/hipotesis conservadora segun esquema_i...


In [11]:
# 7.5. Resultados finales de dimensionado
df_dimensionado


,numero,tipo_tramo,tramo_zona,designacion,nodo_ini,nodo_fin,longitud_m,q_m3_h,q_kg_h,designacion_tubo,...,Lc_m,Q_D,P_ini_rel_bar,P_fin_rel_bar,delta_p_bar,velocidad_ms,cumple_Q_D,cumple_presion,cumple_velocidad,estado
0,1,derivacion_deposito,Derivacion vertical deposito D1,D1-D4,D1,D4,1.90,24.948674,46.953405,18x1,...,2.540,1.559292,1.700000,1.684633,0.015367,13.655955,True,True,True,Conforme
1,1,derivacion_deposito,Derivacion vertical deposito D2,D2-D4,D2,D4,1.90,24.948674,46.953405,18x1,...,2.540,1.559292,1.700000,1.684633,0.015367,13.655955,True,True,True,Conforme
2,1,derivacion_deposito,Derivacion vertical deposito D3,D3-D4,D3,D4,1.90,24.948674,46.953405,18x1,...,2.540,1.559292,1.700000,1.684633,0.015367,13.655955,True,True,True,Conforme
3,1,derivacion_deposito,Derivacion vertical deposito D4,D4 vertical,D4_dep,D4,1.90,24.948674,46.953405,18x1,...,2.540,1.559292,1.700000,1.684633,0.015367,13.655955,True,True,True,Conforme
4,2,red_principal,Colector superior desde zona de depósitos,D4-A,D4,A,15.12,99.794697,187.813620,35x1.5,...,17.040,3.118584,1.700000,1.654244,0.045756,13.811529,True,True,True,Conforme
5,3,red_principal,Derivación hacia C1,A-C1,A,C1,1.81,2.285375,4.301075,15x1,...,2.850,0.175798,1.654244,1.653630,0.000614,1.916924,True,True,True,Conforme
6,4,red_principal,Bajante a C2,A-B,A,B,4.97,97.509322,183.512545,35x1.5,...,6.570,3.047166,1.654244,1.637131,0.017113,13.582372,True,True,True,Conforme
7,5,red_principal,Derivación hacia C2,B-C2,B,C2,21.79,2.285375,4.301075,15x1,...,22.830,0.175798,1.637131,1.632177,0.004954,1.932470,True,True,True,Conforme
8,6,red_principal,Bajante a C3,B-C,B,C,17.32,95.223948,179.211470,42x1.5,...,19.270,2.441640,1.637131,1.618479,0.018652,8.993186,True,True,True,Conforme
9,7,red_principal,Derivación hacia C3,C-C3,C,C3,1.75,19.044790,35.842294,28x1,...,3.830,0.732492,1.618479,1.617075,0.001404,4.049094,True,True,True,Conforme


In [12]:
# 7.6. No conformidades y control manual
print('Resumen agrupado de derivaciones de depósito:')
print(resumen_depositos_agrupado.to_string(index=False))
print('\nControl manual D4-A:')
for clave, valor in control_manual.items():
    print(f'{clave}: {valor}')

if df_no_conformidades.empty:
    print('\nNo se detectan no conformidades.')
else:
    print('\nNo conformidades:')
    print(df_no_conformidades.to_string(index=False))


Resumen agrupado de derivaciones de depósito:
           designacion  unidades  longitud_m_por_unidad  q_m3_h_por_unidad designacion_tubo  D_int_mm  Lc_m_por_unidad  P_fin_rel_bar_min  velocidad_ms_max   estado
D1/D2/D3/D4 verticales         4                    1.9          24.948674             18x1      16.0             2.54           1.684633         13.655955 Conforme

Control manual D4-A:
tramo: D4-A
Lc_recalculada_m: 17.04
Q_D_recalculado: 3.11858428652505
velocidad_recalculada_ms: 13.811529054753628

No se detectan no conformidades.


## 8. Visión final del dimensionado

Resumen final guardado en la libreta para revisión directa sin abrir los ficheros Markdown generados.


In [13]:
# 8. Visión final del dimensionado
columnas_vision = [
    'designacion', 'tipo_tramo', 'longitud_m', 'q_m3_h', 'designacion_tubo',
    'D_int_mm', 'Lc_m', 'P_ini_rel_bar', 'P_fin_rel_bar', 'velocidad_ms', 'estado'
]

print('--- Visión final del dimensionado de tuberías GLP ---')
print(f"Material: {resultados_dimensionado['material_seleccionado']}")
print('Presión inicial: 1.70 bar relativos')
print(f"Caudal punta total: {resultados_dimensionado['q_total_m3_h']:.2f} m3/h ({resultados_dimensionado['q_total_kg_h']:.2f} kg/h)")
print(f"Tramos dimensionados: {len(df_dimensionado)}")
print(f"No conformidades: {len(df_no_conformidades)}")
if df_no_conformidades.empty:
    print('No se detectan no conformidades.')
print(f"Markdown de cálculo: {resultados_dimensionado['salida_markdown']}")
print(f"Anotación técnica: {resultados_dimensionado['salida_anotacion']}")

df_dimensionado[columnas_vision]


--- Visión final del dimensionado de tuberías GLP ---
Material: Cobre duro estirado sin soldadura EN 1057, espesor minimo 1 mm, instalacion aerea protegida
Presión inicial: 1.70 bar relativos
Caudal punta total: 99.79 m3/h (187.81 kg/h)
Tramos dimensionados: 15
No conformidades: 0
No se detectan no conformidades.
Markdown de cálculo: H:\Unidades compartidas\Practicas_Inst2\4_GLPs\calculos\dimensionado_tuberias_glp.md
Anotación técnica: H:\Unidades compartidas\Practicas_Inst2\4_GLPs\Proyecto\Anotaciones\dimensionado_tuberias_glp.md


,designacion,tipo_tramo,longitud_m,q_m3_h,designacion_tubo,D_int_mm,Lc_m,P_ini_rel_bar,P_fin_rel_bar,velocidad_ms,estado
0,D1-D4,derivacion_deposito,1.90,24.948674,18x1,16.0,2.540,1.700000,1.684633,13.655955,Conforme
1,D2-D4,derivacion_deposito,1.90,24.948674,18x1,16.0,2.540,1.700000,1.684633,13.655955,Conforme
2,D3-D4,derivacion_deposito,1.90,24.948674,18x1,16.0,2.540,1.700000,1.684633,13.655955,Conforme
3,D4 vertical,derivacion_deposito,1.90,24.948674,18x1,16.0,2.540,1.700000,1.684633,13.655955,Conforme
4,D4-A,red_principal,15.12,99.794697,35x1.5,32.0,17.040,1.700000,1.654244,13.811529,Conforme
5,A-C1,red_principal,1.81,2.285375,15x1,13.0,2.850,1.654244,1.653630,1.916924,Conforme
6,A-B,red_principal,4.97,97.509322,35x1.5,32.0,6.570,1.654244,1.637131,13.582372,Conforme
7,B-C2,red_principal,21.79,2.285375,15x1,13.0,22.830,1.637131,1.632177,1.932470,Conforme
8,B-C,red_principal,17.32,95.223948,42x1.5,39.0,19.270,1.637131,1.618479,8.993186,Conforme
9,C-C3,red_principal,1.75,19.044790,28x1,26.0,3.830,1.618479,1.617075,4.049094,Conforme
